In [1]:
from google import genai
import os
api_key=os.environ['GOOGLE_API_KEY']
client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash"

d:\gsoc2026\cookbook\venv\lib\site-packages\google\auth\__init__.py:54: FutureWarning: 
    You are using a Python version 3.9 past its end of life. Google will update
    google-auth with critical bug fixes on a best-effort basis, but not
    with any other fixes or features. Please upgrade your Python version,
    and then update google-auth.
    
  warnings.warn(eol_message.format("3.9"), FutureWarning)
d:\gsoc2026\cookbook\venv\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: 
    You are using a Python version 3.9 past its end of life. Google will update
    google-auth with critical bug fixes on a best-effort basis, but not
    with any other fixes or features. Please upgrade your Python version,
    and then update google-auth.
    
  warnings.warn(eol_message.format("3.9"), FutureWarning)


In [2]:
# Short-term memory (verbatim, recent turns)
recent_messages = []

# Long-term memory (compressed summary)
conversation_summary = ""

# Structured ticket state (machine-readable)
ticket_state = {
    "issue": None,
    "status": "open",
    "attempted_fixes": [],
}

In [3]:
SYSTEM_INSTRUCTION = """
You are a customer support assistant helping resolve a technical issue.

Goals:
- Understand the user's issue
- Avoid repeating questions that were already answered
- Do not suggest fixes that were already attempted
- Move the ticket toward resolution

You are given:
- A summary of earlier conversation (long-term memory)
- Current ticket state (structured)
- Recent messages (short-term memory)

Rules:
- Do NOT suggest fixes that were already attempted.
- If an issue is resolved, acknowledge it and do not repeat troubleshooting.
- If the same question is asked again, answer differently based on ticket state.
- If new information contradicts a resolved state, reopen the ticket.
- Be concise and professional.
"""


In [4]:
def build_context():
    return f"""
=== Ticket Summary (Long-Term Memory) ===
{conversation_summary or "No summary yet."}

=== Ticket State (Structured Memory) ===
Issue: {ticket_state["issue"]}
Status: {ticket_state["status"]}
Attempted Fixes: {ticket_state["attempted_fixes"]}

=== Recent Conversation (Short-Term Memory) ===
{recent_messages[-4:]}
"""

In [5]:
def update_ticket_state_from_user(user_message: str):
    global ticket_state

    prompt = f"""
        Analyze the user message and update ticket state if needed.

        User message:
        {user_message}

        Current ticket state:
        {ticket_state}

        Return JSON ONLY with fields to update.
        Possible fields:
        - issue
        - status
        - attempted_fixes (append new ones)
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
    )

    try:
        updates = eval(response.text)
        for key, value in updates.items():
            if key == "attempted_fixes" and isinstance(value, list):
                for fix in value:
                    if fix not in ticket_state["attempted_fixes"]:
                        ticket_state["attempted_fixes"].append(fix)
            else:
                ticket_state[key] = value
    except Exception:
        pass


In [6]:
def update_conversation_summary():
    global conversation_summary

    if not recent_messages:
        return

    prompt = f"""
        Summarize this support conversation.
        Focus on:
        - the original issue
        - fixes attempted
        - whether the issue was resolved

        Conversation:
        {recent_messages}
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
    )

    conversation_summary = response.text.strip()


In [7]:
def update_ticket_state(user_message):
    global ticket_state

    prompt = f"""
        Given the user message below, update ticket fields if applicable.

        User message:
        {user_message}

        Current ticket state:
        {ticket_state}

        Return JSON with any updates, or empty JSON if none.
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
    )

    try:
        updates = eval(response.text)
        for key, value in updates.items():
            ticket_state[key] = value
    except Exception:
        pass


In [8]:
def assistant_reply(user_message: str):
    context = build_context()

    response = client.models.generate_content(
        model=MODEL,
        contents=[
            SYSTEM_INSTRUCTION,
            context,
            f"User message: {user_message}",
        ],
    )

    return response.text.strip()


In [9]:
print("\nContext-Aware Support Ticket Assistant (Initial Session)")
print("Describe your issue. Type 'fixed' once the issue is resolved.\n")

while True:
    user_input = input("User: ")
    print(f"\nUser: {user_input}")
    if user_input.lower() == "quit":
        break

    recent_messages.append(f"User: {user_input}")

    update_ticket_state_from_user(user_input)

    reply = assistant_reply(user_input)
    print("\nAssistant:", reply)
    print("→ Type 'fixed' if the issue is resolved.\n")

    recent_messages.append(f"Assistant: {reply}")

    if "works now" in user_input.lower() or "fixed" in user_input.lower():
        ticket_state["status"] = "resolved"
        update_conversation_summary()
        recent_messages.clear()
        print("\nTicket marked as resolved.\n")
        break



Context-Aware Support Ticket Assistant (Initial Session)
Describe your issue. Type 'fixed' once the issue is resolved.


User: I have trouble logging into my account. It says "Invalid credentials" each time I try to log in

Assistant: I understand you're having trouble logging into your account and are seeing an "Invalid credentials" error.

Could you please confirm that you're using the correct email address or username associated with your account? Also, have you tried using the "Forgot Password" option to reset your password yet?
→ Type 'fixed' if the issue is resolved.


User: Yes I am using the correct username. Let me try the Forgot Password option again. 

Assistant: Thank you for confirming you're using the correct username. Please go ahead and try the "Forgot Password" option again.

Let me know if you encounter any further issues after attempting the password reset, and I'll be happy to assist you further.
→ Type 'fixed' if the issue is resolved.


User: Okay yes, it works n

Note:
This example assumes the user is reopening an existing ticket.
In real systems, ticket routing and creation logic would live outside the model.
The goal here is to demonstrate how persisted context changes assistant behavior.

In [10]:
print("\nContext-Aware Support Ticket Assistant (Reopening Ticket)")
print("The previous ticket is being reopened.\n")

# Explicit reopen (no ambiguity)
ticket_state["status"] = "open"

while True:
    user_input = input("User: ")
    print(f"\nUser: {user_input}")
    if user_input.lower() == "quit":
        break

    recent_messages.append(f"User: {user_input}")

    update_ticket_state_from_user(user_input)

    reply = assistant_reply(user_input)
    print("\nAssistant:", reply)
    print("→ Type 'fixed' if the issue is resolved.\n")

    recent_messages.append(f"Assistant: {reply}")

    if "works now" in user_input.lower() or "fixed" in user_input.lower():
        ticket_state["status"] = "resolved"
        update_conversation_summary()
        recent_messages.clear()
        print("\nTicket re-resolved.\n")
        break



Context-Aware Support Ticket Assistant (Reopening Ticket)
The previous ticket is being reopened.


User: The issue came again this morning. It says "system ran into an error, please try again" each time I try to log in

Assistant: I'm sorry to hear you're experiencing login issues again. It sounds like you're encountering a new error message: "system ran into an error, please try again."

This is different from the "Invalid credentials" error you previously saw. To help me understand this new issue, could you please tell me:

1.  Are you using the same device and browser as when you successfully logged in after resetting your password?
2.  Have you tried clearing your browser's cache and cookies, or trying a different browser/incognito window?
→ Type 'fixed' if the issue is resolved.


User: Yes, im using the same device and browser. Let me try clearning my browser's cache, I havent done that.

Assistant: Thank you for confirming. Please go ahead and try clearing your browser's cache 